# SAM3 Few-Shot Metric Conversion

This notebook converts SAM3 few-shot COCO instance predictions into dense binary masks so they can be compared with U-Net/ResNet34 using the same pixel-level metrics.

Workflow:

1. Read each run's `test/_annotations.coco.json` ground truth.
2. Read each run's `dumps/ttd/test/coco_predictions_segm.json` predictions.
3. For each image, union all SAM3 predicted instance masks above a score threshold.
4. Compute pixel-level IoU, F1, precision, and recall.
5. Report all-image, positive-only, negative-only, and global pixel metrics.

Run this after SAM3 train/eval jobs have produced prediction JSON files.


In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/users/7/yu001011/csci5527")
SAM3_WORK_ROOT = PROJECT_ROOT / "CSCI5527-final" / "SAM3"

EXPERIMENT_GROUPS = {
    "single_tb": "Single-TB",
    "shift_ta_tc_to_tb_10pct": "Shift-TA_TC-to-TB_10pct",
    "shift_ta_tb_to_tc_10pct": "Shift-TA_TB-to-TC_10pct",
}

# Keep the same order as the batch training notebook: Single-TB first, then shift experiments.
BATCH_GROUP_ORDER = [
    "single_tb",
    "shift_ta_tc_to_tb_10pct",
    "shift_ta_tb_to_tc_10pct",
]
BATCH_SHOTS = [5, 10, 25, 50]
RUN_TAG = "stable_lowlr_v1"

# Threshold sweep. You can add/remove values here.
SCORE_THRESHOLDS = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50]

OUTPUT_DIR = SAM3_WORK_ROOT / "outputs" / "fewshot_metric_conversion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = []
for group_key in BATCH_GROUP_ORDER:
    experiment_name = EXPERIMENT_GROUPS[group_key]
    for shot in BATCH_SHOTS:
        key = f"{group_key}_{shot}shot_{RUN_TAG}"
        run_root = SAM3_WORK_ROOT / "fewshot_data" / experiment_name / f"{shot}_shot_per_class"
        run_dir = SAM3_WORK_ROOT / "outputs" / "fewshot_runs" / experiment_name / f"{shot}_shot_per_class_{RUN_TAG}"
        EXPERIMENTS.append({
            "experiment_key": key,
            "experiment_name": experiment_name,
            "shot_per_class": shot,
            "run_tag": RUN_TAG,
            "gt_json": run_root / "test" / "_annotations.coco.json",
            "pred_json": run_dir / "dumps" / "ttd" / "test" / "coco_predictions_segm.json",
        })

pd.DataFrame([
    {
        "experiment_key": e["experiment_key"],
        "experiment_name": e["experiment_name"],
        "shot_per_class": e["shot_per_class"],
        "gt_exists": e["gt_json"].exists(),
        "pred_exists": e["pred_json"].exists(),
        "pred_json": str(e["pred_json"]),
    }
    for e in EXPERIMENTS
])


,experiment_key,experiment_name,shot_per_class,gt_exists,pred_exists,pred_json
0,single_tb_5shot_stable_lowlr_v1,Single-TB,5,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
1,single_tb_10shot_stable_lowlr_v1,Single-TB,10,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
2,single_tb_25shot_stable_lowlr_v1,Single-TB,25,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
3,single_tb_50shot_stable_lowlr_v1,Single-TB,50,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
4,shift_ta_tc_to_tb_10pct_5shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,5,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
5,shift_ta_tc_to_tb_10pct_10shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,10,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
6,shift_ta_tc_to_tb_10pct_25shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,25,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
7,shift_ta_tc_to_tb_10pct_50shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,50,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
8,shift_ta_tb_to_tc_10pct_5shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,5,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
9,shift_ta_tb_to_tc_10pct_10shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,10,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...


In [2]:
def _decode_compressed_rle_counts(counts_text):
    """Decode COCO compressed RLE counts into run lengths.

    This avoids requiring pycocotools just for metric conversion. It supports the
    compressed RLE strings used in the SAM3 prediction and GT JSON files.
    """
    if isinstance(counts_text, list):
        return counts_text
    if isinstance(counts_text, bytes):
        counts_text = counts_text.decode("ascii")

    counts = []
    pos = 0
    count_index = 0
    while pos < len(counts_text):
        value = 0
        shift = 0
        while True:
            char_value = ord(counts_text[pos]) - 48
            pos += 1
            value |= (char_value & 0x1F) << shift
            shift += 5
            if not (char_value & 0x20):
                if char_value & 0x10:
                    value |= -1 << shift
                break
        if count_index > 2:
            value += counts[count_index - 2]
        counts.append(value)
        count_index += 1
    return counts


def decode_coco_rle(rle):
    """Decode a COCO RLE object into a boolean H x W mask."""
    height, width = rle["size"]
    counts = _decode_compressed_rle_counts(rle["counts"])
    flat = np.zeros(height * width, dtype=bool)
    index = 0
    value = False
    for run_length in counts:
        if run_length < 0:
            raise ValueError(f"Invalid negative RLE run length: {run_length}")
        if value and run_length:
            flat[index:index + run_length] = True
        index += run_length
        value = not value
    return flat.reshape((height, width), order="F")


def pixel_stats(pred_mask, true_mask, smooth=1e-6):
    pred = pred_mask.astype(bool)
    true = true_mask.astype(bool)

    tp = float(np.logical_and(pred, true).sum())
    fp = float(np.logical_and(pred, np.logical_not(true)).sum())
    fn = float(np.logical_and(np.logical_not(pred), true).sum())
    tn = float(np.logical_and(np.logical_not(pred), np.logical_not(true)).sum())

    iou = (tp + smooth) / (tp + fp + fn + smooth)
    precision = (tp + smooth) / (tp + fp + smooth)
    recall = (tp + smooth) / (tp + fn + smooth)
    f1 = 2.0 * precision * recall / (precision + recall + smooth)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "iou": iou,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


def safe_mean(values):
    values = [v for v in values if not pd.isna(v)]
    return float(np.mean(values)) if values else np.nan


def summarize_per_image(per_image_df):
    positive = per_image_df[per_image_df["has_gt_crack"]]
    negative = per_image_df[~per_image_df["has_gt_crack"]]

    total_tp = per_image_df["tp"].sum()
    total_fp = per_image_df["fp"].sum()
    total_fn = per_image_df["fn"].sum()
    global_iou = (total_tp + 1e-6) / (total_tp + total_fp + total_fn + 1e-6)
    global_precision = (total_tp + 1e-6) / (total_tp + total_fp + 1e-6)
    global_recall = (total_tp + 1e-6) / (total_tp + total_fn + 1e-6)
    global_f1 = 2 * global_precision * global_recall / (global_precision + global_recall + 1e-6)

    return {
        "num_test": int(len(per_image_df)),
        "num_positive": int(len(positive)),
        "num_negative": int(len(negative)),
        "test_iou": safe_mean(per_image_df["iou"]),
        "test_f1": safe_mean(per_image_df["f1"]),
        "test_precision": safe_mean(per_image_df["precision"]),
        "test_recall": safe_mean(per_image_df["recall"]),
        "positive_iou": safe_mean(positive["iou"]),
        "positive_f1": safe_mean(positive["f1"]),
        "positive_precision": safe_mean(positive["precision"]),
        "positive_recall": safe_mean(positive["recall"]),
        "negative_clean_rate": float((negative["pred_pixels"] == 0).mean()) if len(negative) else np.nan,
        "negative_fp_rate": float((negative["pred_pixels"] > 0).mean()) if len(negative) else np.nan,
        "mean_num_predictions": safe_mean(per_image_df["num_predictions"]),
        "mean_gt_pixels": safe_mean(per_image_df["gt_pixels"]),
        "mean_pred_pixels": safe_mean(per_image_df["pred_pixels"]),
        "global_iou": float(global_iou),
        "global_f1": float(global_f1),
        "global_precision": float(global_precision),
        "global_recall": float(global_recall),
    }


In [3]:
def load_ground_truth_masks(gt_json_path):
    gt = json.loads(Path(gt_json_path).read_text())
    image_info = {image["id"]: image for image in gt["images"]}
    gt_masks = {
        image_id: np.zeros((image["height"], image["width"]), dtype=bool)
        for image_id, image in image_info.items()
    }

    for ann in gt.get("annotations", []):
        segmentation = ann.get("segmentation")
        if not isinstance(segmentation, dict):
            raise NotImplementedError(
                "This notebook expects RLE segmentations. If your JSON uses polygons, "
                "run it in an environment with pycocotools and add polygon conversion."
            )
        gt_masks[ann["image_id"]] |= decode_coco_rle(segmentation)

    return gt, image_info, gt_masks


def convert_one_experiment(experiment, score_threshold):
    gt_json_path = experiment["gt_json"]
    pred_json_path = experiment["pred_json"]

    if not gt_json_path.exists():
        raise FileNotFoundError(f"Missing ground-truth JSON: {gt_json_path}")
    if not pred_json_path.exists() or pred_json_path.stat().st_size <= 2:
        raise FileNotFoundError(f"Missing or empty prediction JSON: {pred_json_path}")

    _, image_info, gt_masks = load_ground_truth_masks(gt_json_path)
    pred_masks = {image_id: np.zeros_like(mask) for image_id, mask in gt_masks.items()}
    num_predictions = {image_id: 0 for image_id in gt_masks}

    predictions = json.loads(pred_json_path.read_text())
    for pred_ann in predictions:
        image_id = pred_ann.get("image_id")
        if image_id not in pred_masks:
            continue
        if float(pred_ann.get("score", 0.0)) < score_threshold:
            continue
        segmentation = pred_ann.get("segmentation")
        if not isinstance(segmentation, dict):
            continue
        pred_masks[image_id] |= decode_coco_rle(segmentation)
        num_predictions[image_id] += 1

    rows = []
    for image_id in sorted(gt_masks):
        gt_mask = gt_masks[image_id]
        pred_mask = pred_masks[image_id]
        stats = pixel_stats(pred_mask, gt_mask)
        image = image_info[image_id]
        rows.append({
            "experiment_key": experiment["experiment_key"],
            "experiment_name": experiment["experiment_name"],
            "shot_per_class": experiment["shot_per_class"],
            "run_tag": experiment["run_tag"],
            "score_threshold": score_threshold,
            "image_id": image_id,
            "file_name": image.get("file_name", ""),
            "has_gt_crack": bool(gt_mask.any()),
            "has_prediction": bool(pred_mask.any()),
            "gt_pixels": int(gt_mask.sum()),
            "pred_pixels": int(pred_mask.sum()),
            "num_predictions": int(num_predictions[image_id]),
            **stats,
        })

    per_image_df = pd.DataFrame(rows)
    summary = summarize_per_image(per_image_df)
    summary.update({
        "experiment_key": experiment["experiment_key"],
        "experiment_name": experiment["experiment_name"],
        "shot_per_class": experiment["shot_per_class"],
        "run_tag": experiment["run_tag"],
        "score_threshold": score_threshold,
        "gt_json": str(gt_json_path),
        "pred_json": str(pred_json_path),
    })
    return summary, per_image_df

print("Conversion helpers ready.")


Conversion helpers ready.


In [4]:
summary_rows = []
per_image_frames = []
skipped_rows = []

for experiment in EXPERIMENTS:
    print("\n" + "=" * 100)
    print(f"Converting {experiment['experiment_key']}")
    print("Prediction JSON:", experiment["pred_json"])

    if not experiment["pred_json"].exists() or experiment["pred_json"].stat().st_size <= 2:
        print("Skipping: prediction JSON does not exist yet or is empty.")
        skipped_rows.append({
            "experiment_key": experiment["experiment_key"],
            "reason": "missing_or_empty_prediction_json",
            "pred_json": str(experiment["pred_json"]),
        })
        continue

    for score_threshold in SCORE_THRESHOLDS:
        summary, per_image_df = convert_one_experiment(experiment, score_threshold)
        summary_rows.append(summary)
        per_image_frames.append(per_image_df)
        print(
            f"threshold={score_threshold:.2f} "
            f"test_iou={summary['test_iou']:.4f} "
            f"test_f1={summary['test_f1']:.4f} "
            f"positive_iou={summary['positive_iou']:.4f} "
            f"positive_f1={summary['positive_f1']:.4f} "
            f"negative_fp_rate={summary['negative_fp_rate']:.4f}"
        )

summary_df = pd.DataFrame(summary_rows)
per_image_df = pd.concat(per_image_frames, ignore_index=True) if per_image_frames else pd.DataFrame()
skipped_df = pd.DataFrame(skipped_rows)

summary_csv = OUTPUT_DIR / "sam3_fewshot_pixel_metric_summary_threshold_sweep.csv"
per_image_csv = OUTPUT_DIR / "sam3_fewshot_pixel_metric_per_image_threshold_sweep.csv"
skipped_csv = OUTPUT_DIR / "sam3_fewshot_pixel_metric_skipped_runs.csv"

summary_df.to_csv(summary_csv, index=False)
per_image_df.to_csv(per_image_csv, index=False)
skipped_df.to_csv(skipped_csv, index=False)

print("\nSaved summary:", summary_csv)
print("Saved per-image metrics:", per_image_csv)
print("Saved skipped-runs list:", skipped_csv)

display(summary_df)
if len(skipped_df):
    print("Skipped runs:")
    display(skipped_df)



Converting single_tb_5shot_stable_lowlr_v1
Prediction JSON: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Single-TB/5_shot_per_class_stable_lowlr_v1/dumps/ttd/test/coco_predictions_segm.json
threshold=0.00 test_iou=0.0162 test_f1=0.0271 positive_iou=0.0323 positive_f1=0.0541 negative_fp_rate=1.0000
threshold=0.01 test_iou=0.0162 test_f1=0.0271 positive_iou=0.0323 positive_f1=0.0541 negative_fp_rate=1.0000
threshold=0.02 test_iou=0.0162 test_f1=0.0271 positive_iou=0.0324 positive_f1=0.0542 negative_fp_rate=1.0000
threshold=0.05 test_iou=0.1211 test_f1=0.1327 positive_iou=0.0338 positive_f1=0.0570 negative_fp_rate=0.7917
threshold=0.10 test_iou=0.2104 test_f1=0.2264 positive_iou=0.0459 positive_f1=0.0777 negative_fp_rate=0.6250
threshold=0.20 test_iou=0.3998 test_f1=0.4288 positive_iou=0.0913 positive_f1=0.1494 negative_fp_rate=0.2917
threshold=0.30 test_iou=0.4697 test_f1=0.5110 positive_iou=0.1478 positive_f1=0.2303 negative_fp_rate=0.2083
threshold=0.50 test_iou

threshold=0.50 test_iou=0.5814 test_f1=0.6291 positive_iou=0.2045 positive_f1=0.2998 negative_fp_rate=0.0417

Converting shift_ta_tc_to_tb_10pct_50shot_stable_lowlr_v1
Prediction JSON: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Shift-TA_TC-to-TB_10pct/50_shot_per_class_stable_lowlr_v1/dumps/ttd/test/coco_predictions_segm.json
threshold=0.00 test_iou=0.0329 test_f1=0.0534 positive_iou=0.0658 positive_f1=0.1069 negative_fp_rate=1.0000
threshold=0.01 test_iou=0.3657 test_f1=0.3857 positive_iou=0.0647 positive_f1=0.1048 negative_fp_rate=0.3333
threshold=0.02 test_iou=0.3865 test_f1=0.4066 positive_iou=0.0647 positive_f1=0.1048 negative_fp_rate=0.2917
threshold=0.05 test_iou=0.4100 test_f1=0.4319 positive_iou=0.0701 positive_f1=0.1138 negative_fp_rate=0.2500
threshold=0.10 test_iou=0.4490 test_f1=0.4803 positive_iou=0.1064 positive_f1=0.1690 negative_fp_rate=0.2083
threshold=0.20 test_iou=0.5134 test_f1=0.5543 positive_iou=0.1518 positive_f1=0.2337 negative_fp_rate=

,num_test,num_positive,num_negative,test_iou,test_f1,test_precision,test_recall,positive_iou,positive_f1,positive_precision,...,global_f1,global_precision,global_recall,experiment_key,experiment_name,shot_per_class,run_tag,score_threshold,gt_json,pred_json
0,48,24,24,0.016173,0.027071,0.016283,0.971161,0.032346,0.054142,0.032567,...,0.011796,0.005934,0.978246,single_tb_5shot_stable_lowlr_v1,Single-TB,5,stable_lowlr_v1,0.00,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
1,48,24,24,0.016173,0.027071,0.016283,0.971161,0.032346,0.054142,0.032567,...,0.011854,0.005963,0.978246,single_tb_5shot_stable_lowlr_v1,Single-TB,5,stable_lowlr_v1,0.01,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
2,48,24,24,0.016196,0.027117,0.016306,0.971161,0.032392,0.054234,0.032613,...,0.015188,0.007653,0.978246,single_tb_5shot_stable_lowlr_v1,Single-TB,5,stable_lowlr_v1,0.02,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
3,48,24,24,0.121076,0.132689,0.121191,0.960470,0.033820,0.057044,0.034048,...,0.026577,0.013472,0.974805,single_tb_5shot_stable_lowlr_v1,Single-TB,5,stable_lowlr_v1,0.05,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
4,48,24,24,0.210429,0.226364,0.231422,0.891071,0.045859,0.077728,0.087844,...,0.061366,0.031697,0.959437,single_tb_5shot_stable_lowlr_v1,Single-TB,5,stable_lowlr_v1,0.10,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,38,19,19,0.514517,0.531601,0.834455,0.669274,0.081665,0.115834,0.721542,...,0.144297,0.083192,0.543504,shift_ta_tb_to_tc_10pct_50shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,50,stable_lowlr_v1,0.05,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
92,38,19,19,0.515779,0.537317,0.870920,0.618144,0.084190,0.127265,0.794471,...,0.275043,0.200519,0.437729,shift_ta_tb_to_tc_10pct_50shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,50,stable_lowlr_v1,0.10,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
93,38,19,19,0.522285,0.542838,0.909930,0.578597,0.097202,0.138307,0.872492,...,0.332674,0.440514,0.267251,shift_ta_tb_to_tc_10pct_50shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,50,stable_lowlr_v1,0.20,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
94,38,19,19,0.550985,0.571756,0.944274,0.572398,0.101970,0.143512,0.888549,...,0.335902,0.564755,0.239039,shift_ta_tb_to_tc_10pct_50shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,50,stable_lowlr_v1,0.30,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...


## Pick One Threshold Per Experiment

For the final report, avoid selecting the best threshold directly on the test set if you want a strict experimental protocol. A good default is:

- Use one fixed threshold for all SAM3 runs, such as `0.10` or `0.20`, or
- Select threshold using validation predictions, then apply that threshold to test.

The cell below is a convenience view that chooses the best threshold by `positive_f1` from the test sweep. Use it for exploration, not as the strict final result unless you clearly state that threshold was selected post hoc.


In [5]:
if summary_df.empty:
    raise RuntimeError("Run the threshold-sweep conversion cell first.")

best_by_positive_f1 = (
    summary_df.sort_values(["experiment_key", "positive_f1", "positive_iou"], ascending=[True, False, False])
    .groupby("experiment_key", as_index=False)
    .head(1)
    .sort_values(["experiment_name", "shot_per_class"])
)

best_csv = OUTPUT_DIR / "sam3_fewshot_pixel_metric_best_test_positive_f1.csv"
best_by_positive_f1.to_csv(best_csv, index=False)
print("Saved exploratory best-threshold table:", best_csv)

display(best_by_positive_f1[[
    "experiment_name",
    "shot_per_class",
    "score_threshold",
    "test_iou",
    "test_f1",
    "test_precision",
    "test_recall",
    "positive_iou",
    "positive_f1",
    "positive_precision",
    "positive_recall",
    "negative_fp_rate",
    "mean_num_predictions",
]])


Saved exploratory best-threshold table: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_metric_conversion/sam3_fewshot_pixel_metric_best_test_positive_f1.csv


,experiment_name,shot_per_class,score_threshold,test_iou,test_f1,test_precision,test_recall,positive_iou,positive_f1,positive_precision,positive_recall,negative_fp_rate,mean_num_predictions
69,Shift-TA_TB-to-TC_10pct,5,0.2,0.561009,0.600510,0.751981,0.768151,0.174650,0.253652,0.556593,0.536302,0.052632,0.973684
76,Shift-TA_TB-to-TC_10pct,10,0.1,0.544883,0.580034,0.682608,0.823913,0.195029,0.265331,0.470480,0.647827,0.105263,4.526316
85,Shift-TA_TB-to-TC_10pct,25,0.2,0.550895,0.577672,0.885534,0.632363,0.154422,0.207976,0.823700,0.264725,0.052632,0.552632
94,Shift-TA_TB-to-TC_10pct,50,0.3,0.550985,0.571756,0.944274,0.572398,0.101970,0.143512,0.888549,0.144795,0.000000,0.157895
39,Shift-TA_TC-to-TB_10pct,5,0.5,0.587649,0.629238,0.785871,0.757460,0.175297,0.258476,0.571742,0.514920,0.000000,0.520833
47,Shift-TA_TC-to-TB_10pct,10,0.5,0.601504,0.648466,0.752104,0.804047,0.203009,0.296933,0.504209,0.608093,0.000000,0.750000
55,Shift-TA_TC-to-TB_10pct,25,0.5,0.581442,0.629089,0.732491,0.799907,0.204550,0.299845,0.506649,0.599813,0.041667,0.958333
63,Shift-TA_TC-to-TB_10pct,50,0.5,0.581110,0.636932,0.717624,0.820072,0.245552,0.357198,0.518582,0.640144,0.083333,1.041667
7,Single-TB,5,0.5,0.589251,0.631453,0.787449,0.757423,0.178502,0.262907,0.574899,0.514845,0.000000,0.500000
15,Single-TB,10,0.5,0.597862,0.644785,0.774002,0.795378,0.195723,0.289571,0.548004,0.590755,0.000000,0.625000


## Optional: U-Net Positive/Negative Test Metrics

The U-Net baseline CSV only stores overall test metrics. Run the next cell if you want to recompute saved U-Net+ResNet34 checkpoints on the same test splits and add:

- `positive_iou`
- `positive_f1`
- `negative_fp_rate`

This makes the U-Net rows easier to compare with SAM3 zero-shot and few-shot. The cell requires the original baseline environment (`torch`, `fastai`, `segmentation_models_pytorch`, and `PIL`). If those packages are unavailable, skip this cell; the final comparison will still run, but U-Net positive-only columns will remain empty.


In [ ]:
RUN_UNET_POSITIVE_NEGATIVE_METRICS = True
UNET_MODEL_DIR = PROJECT_ROOT / "CSCI5527-final" / "baseline_models" / "models"
UNET_DETAIL_CSV = OUTPUT_DIR / "unet_resnet34_imagenet_positive_negative_test_metrics.csv"
UNET_PER_IMAGE_CSV = OUTPUT_DIR / "unet_resnet34_imagenet_per_image_test_metrics.csv"

if not RUN_UNET_POSITIVE_NEGATIVE_METRICS:
    print("Skipping U-Net positive/negative metric recomputation.")
elif UNET_DETAIL_CSV.exists():
    print("Using existing U-Net positive/negative metric CSV:", UNET_DETAIL_CSV)
    display(pd.read_csv(UNET_DETAIL_CSV))
else:
    import sys
    sys.path.insert(0, str(PROJECT_ROOT / "CSCI5527-final"))

    try:
        import torch
        import segmentation_models_pytorch as smp
        from preprocessing import TunnelDataPipeline
    except Exception as exc:
        print("Could not import the baseline inference dependencies.")
        print("Install/use the baseline environment, then rerun this cell if you need U-Net positive-only metrics.")
        print("Import error:", repr(exc))
    else:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Using device:", device)

        baseline_experiments = {
            "Single-TB": {
                "train_files": ["TB_train.csv"],
                "val_files": ["TB_val.csv"],
                "test_files": ["TB_test.csv"],
            },
            "Shift-TA_TC-to-TB_10pct": {
                "train_files": ["TA_train.csv", "TC_train.csv"],
                "val_files": ["TA_val.csv", "TC_val.csv"],
                "test_files": ["TB_test.csv"],
            },
            "Shift-TA_TB-to-TC_10pct": {
                "train_files": ["TA_train.csv", "TB_train.csv"],
                "val_files": ["TA_val.csv", "TB_val.csv"],
                "test_files": ["TC_test.csv"],
            },
        }

        def build_baseline_test_loader(config):
            dataset_folder = PROJECT_ROOT / "CSCI5527-final" / "TACK_Tunnel_Data"
            csv_source_dir = dataset_folder / "2_model_input"
            raw_mask_dir = dataset_folder / "3_mask"
            pipeline = TunnelDataPipeline(base_dir=str(dataset_folder), original_mask_dir=str(raw_mask_dir))
            df_train_val, df_test = pipeline.load_csv_data(
                csv_source_dir=str(csv_source_dir),
                train_files=config["train_files"],
                val_files=config["val_files"],
                test_files=config["test_files"],
            )
            df_train_val_ready = pipeline.sanitize_masks(df_train_val, class_pixel_value=40)
            df_test_ready = pipeline.sanitize_masks(df_test, class_pixel_value=40)
            _, _, test_dl = pipeline.get_dataloaders(
                train_val_df=df_train_val_ready,
                test_df=df_test_ready,
                bs=16,
                img_size=512,
                custom_stats=None,
            )
            return test_dl

        def per_sample_pixel_stats(pred_mask, true_mask, smooth=1e-6):
            pred = pred_mask.bool()
            true = true_mask.bool()
            tp = torch.logical_and(pred, true).sum().float().item()
            fp = torch.logical_and(pred, torch.logical_not(true)).sum().float().item()
            fn = torch.logical_and(torch.logical_not(pred), true).sum().float().item()
            tn = torch.logical_and(torch.logical_not(pred), torch.logical_not(true)).sum().float().item()
            iou = (tp + smooth) / (tp + fp + fn + smooth)
            precision = (tp + smooth) / (tp + fp + smooth)
            recall = (tp + smooth) / (tp + fn + smooth)
            f1 = 2.0 * precision * recall / (precision + recall + smooth)
            return {
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "tn": tn,
                "iou": iou,
                "f1": f1,
                "precision": precision,
                "recall": recall,
            }

        summary_rows = []
        per_image_rows = []

        for experiment_name, config in baseline_experiments.items():
            model_run = f"Unet-resnet34-imagenet_{experiment_name}"
            ckpt_path = UNET_MODEL_DIR / f"{model_run}.pth"
            print("\n" + "=" * 100)
            print("Evaluating:", model_run)
            print("Checkpoint:", ckpt_path)

            if not ckpt_path.exists():
                print("Skipping: checkpoint not found.")
                continue

            test_dl = build_baseline_test_loader(config)
            model = smp.Unet("resnet34", encoder_weights="imagenet", classes=2)
            state = torch.load(ckpt_path, map_location=device)
            model.load_state_dict(state)
            model.to(device)
            model.eval()

            image_index = 0
            with torch.no_grad():
                for images, masks in test_dl:
                    images = torch.as_tensor(images, device=device)
                    masks = torch.as_tensor(masks, device=device).long()
                    if masks.ndim == 4:
                        masks = masks.squeeze(1)
                    outputs = model(images)
                    preds = outputs.argmax(dim=1)

                    for batch_idx in range(preds.shape[0]):
                        pred_mask = preds[batch_idx].detach().cpu()
                        true_mask = masks[batch_idx].detach().cpu()
                        stats = per_sample_pixel_stats(pred_mask == 1, true_mask == 1)
                        gt_pixels = int((true_mask == 1).sum().item())
                        pred_pixels = int((pred_mask == 1).sum().item())
                        row = {
                            "model": "U-Net+ResNet34 ImageNet",
                            "model_run": model_run,
                            "experiment_name": experiment_name,
                            "image_index": image_index,
                            "has_gt_crack": gt_pixels > 0,
                            "has_prediction": pred_pixels > 0,
                            "gt_pixels": gt_pixels,
                            "pred_pixels": pred_pixels,
                            **stats,
                        }
                        per_image_rows.append(row)
                        image_index += 1

            exp_df = pd.DataFrame([r for r in per_image_rows if r["model_run"] == model_run])
            positive = exp_df[exp_df["has_gt_crack"]]
            negative = exp_df[~exp_df["has_gt_crack"]]
            summary = {
                "model": "U-Net+ResNet34 ImageNet",
                "model_run": model_run,
                "experiment_name": experiment_name,
                "num_test": int(len(exp_df)),
                "num_positive": int(len(positive)),
                "num_negative": int(len(negative)),
                "positive_iou": float(positive["iou"].mean()) if len(positive) else np.nan,
                "positive_f1": float(positive["f1"].mean()) if len(positive) else np.nan,
                "positive_precision": float(positive["precision"].mean()) if len(positive) else np.nan,
                "positive_recall": float(positive["recall"].mean()) if len(positive) else np.nan,
                "negative_fp_rate": float((negative["pred_pixels"] > 0).mean()) if len(negative) else np.nan,
                "negative_clean_rate": float((negative["pred_pixels"] == 0).mean()) if len(negative) else np.nan,
                "per_image_test_iou": float(exp_df["iou"].mean()),
                "per_image_test_f1": float(exp_df["f1"].mean()),
            }
            summary_rows.append(summary)
            print(
                f"positive_iou={summary['positive_iou']:.4f}, "
                f"positive_f1={summary['positive_f1']:.4f}, "
                f"negative_fp_rate={summary['negative_fp_rate']:.4f}"
            )

        unet_detail_df = pd.DataFrame(summary_rows)
        unet_per_image_df = pd.DataFrame(per_image_rows)
        unet_detail_df.to_csv(UNET_DETAIL_CSV, index=False)
        unet_per_image_df.to_csv(UNET_PER_IMAGE_CSV, index=False)
        print("\nSaved U-Net summary:", UNET_DETAIL_CSV)
        print("Saved U-Net per-image metrics:", UNET_PER_IMAGE_CSV)
        display(unet_detail_df)


## Compare With U-Net Baseline and SAM3 Zero-Shot

This optional cell loads existing U-Net/ResNet34 baseline results and SAM3 zero-shot results, then combines them with the SAM3 few-shot conversion table.

Zero-shot rows are read from `outputs/zero_shot/sam3_zero_shot_batch_baseline_compatible_summary.csv`. Positive-only IoU/F1 and negative false-positive rate are computed from each zero-shot per-image CSV when available.


In [ ]:
BASELINE_CSV = PROJECT_ROOT / "CSCI5527-final" / "baseline_models" / "TTD_baseline_results.csv"
UNET_DETAIL_CSV = OUTPUT_DIR / "unet_resnet34_imagenet_positive_negative_test_metrics.csv"
ZERO_SHOT_SUMMARY_CSV = SAM3_WORK_ROOT / "outputs" / "zero_shot" / "sam3_zero_shot_batch_baseline_compatible_summary.csv"

wanted = ["Single-TB", "Shift-TA_TC-to-TB_10pct", "Shift-TA_TB-to-TC_10pct"]
compare_cols = [
    "model",
    "model_run",
    "experiment_name",
    "shot_per_class",
    "score_threshold",
    "test_iou",
    "test_f1",
    "test_precision",
    "test_recall",
    "positive_iou",
    "positive_f1",
    "negative_fp_rate",
]

compare_frames = []

# U-Net / ResNet34 baseline.
if not BASELINE_CSV.exists():
    print("Baseline CSV not found:", BASELINE_CSV)
else:
    baseline_df = pd.read_csv(BASELINE_CSV)
    baseline_df["experiment_name"] = baseline_df["Experiment"].str.replace("Unet-resnet34-imagenet_", "", regex=False)
    baseline_subset = baseline_df[baseline_df["experiment_name"].isin(wanted)].copy()
    baseline_subset = baseline_subset.rename(columns={
        "Experiment": "model_run",
        "Test_IoU": "test_iou",
        "Test_F1": "test_f1",
        "Test_Prec": "test_precision",
        "Test_Recall": "test_recall",
    })
    baseline_subset["model"] = "U-Net+ResNet34 ImageNet"
    baseline_subset["shot_per_class"] = "full baseline"
    baseline_subset["score_threshold"] = np.nan
    baseline_subset["positive_iou"] = np.nan
    baseline_subset["positive_f1"] = np.nan
    baseline_subset["negative_fp_rate"] = np.nan

    if UNET_DETAIL_CSV.exists():
        unet_detail = pd.read_csv(UNET_DETAIL_CSV)
        fill_cols = ["experiment_name", "positive_iou", "positive_f1", "negative_fp_rate"]
        baseline_subset = baseline_subset.drop(columns=["positive_iou", "positive_f1", "negative_fp_rate"]).merge(
            unet_detail[fill_cols],
            on="experiment_name",
            how="left",
        )
    else:
        print("U-Net positive/negative detail CSV not found; U-Net positive-only columns will be empty:", UNET_DETAIL_CSV)

    compare_frames.append(baseline_subset[compare_cols])

# SAM3 zero-shot baseline. Per-image CSVs let us add positive-only and negative-FP summaries.
def summarize_zero_shot_per_image(per_image_csv):
    if not isinstance(per_image_csv, str) or not per_image_csv:
        return {"positive_iou": np.nan, "positive_f1": np.nan, "negative_fp_rate": np.nan}
    path = Path(per_image_csv)
    if not path.exists():
        return {"positive_iou": np.nan, "positive_f1": np.nan, "negative_fp_rate": np.nan}

    per_image = pd.read_csv(path)
    if "target" in per_image.columns:
        positive = per_image[per_image["target"].astype(int) == 1]
        negative = per_image[per_image["target"].astype(int) == 0]
    elif "label" in per_image.columns:
        positive = per_image[per_image["label"].astype(str) == "crack"]
        negative = per_image[per_image["label"].astype(str) != "crack"]
    else:
        return {"positive_iou": np.nan, "positive_f1": np.nan, "negative_fp_rate": np.nan}

    # For empty-GT negative images, the existing metric code gives IoU ~= 1 when
    # the prediction is also empty. Treat noticeably lower IoU as a false positive.
    negative_fp_rate = float((negative["iou"] < 0.999).mean()) if len(negative) else np.nan
    return {
        "positive_iou": float(positive["iou"].mean()) if len(positive) else np.nan,
        "positive_f1": float(positive["f1"].mean()) if len(positive) else np.nan,
        "negative_fp_rate": negative_fp_rate,
    }

if not ZERO_SHOT_SUMMARY_CSV.exists():
    print("Zero-shot summary CSV not found:", ZERO_SHOT_SUMMARY_CSV)
else:
    zero_df = pd.read_csv(ZERO_SHOT_SUMMARY_CSV)
    zero_df = zero_df[zero_df["Experiment"].isin(wanted)].copy()
    zero_extra = zero_df["Per_Image_Results_CSV"].apply(summarize_zero_shot_per_image).apply(pd.Series)
    zero_df = pd.concat([zero_df.reset_index(drop=True), zero_extra.reset_index(drop=True)], axis=1)
    zero_df = zero_df.rename(columns={
        "Experiment": "experiment_name",
        "Test_IoU": "test_iou",
        "Test_F1": "test_f1",
        "Test_Prec": "test_precision",
        "Test_Recall": "test_recall",
        "ConfidenceThreshold": "score_threshold",
    })
    zero_df["model"] = "SAM3 zero-shot"
    zero_df["model_run"] = zero_df["Model"] + "_" + zero_df["experiment_name"]
    zero_df["shot_per_class"] = "zero-shot"
    compare_frames.append(zero_df[compare_cols])

# SAM3 few-shot converted metrics. This is the exploratory best-threshold view.
if "best_by_positive_f1" not in globals() or best_by_positive_f1.empty:
    print("Few-shot best-threshold table is not available. Run the previous cell first.")
else:
    sam3_compare = best_by_positive_f1.copy()
    sam3_compare["model"] = "SAM3 few-shot"
    sam3_compare["model_run"] = sam3_compare["experiment_key"]
    compare_frames.append(sam3_compare[compare_cols])

if not compare_frames:
    raise RuntimeError("No comparison sources were available.")

comparison_df = pd.concat(compare_frames, ignore_index=True)
comparison_csv = OUTPUT_DIR / "sam3_vs_unet_pixel_metric_comparison.csv"
comparison_df.to_csv(comparison_csv, index=False)
print("Saved comparison table:", comparison_csv)

display(comparison_df.sort_values(["experiment_name", "model", "shot_per_class"]))
